<a href="https://colab.research.google.com/github/ernestoaguaysol-unpaz/sistemas-inteligentes-2026/blob/main/03_redes_convolucionales/004_CNN_clasificacion_cifar10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import requests
from PIL import Image
from io import BytesIO

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

# Setear globalmente los estilos de matplotlib
sns.set()

# Cargar y explorar dataset CIFAR

In [ ]:
# Algunas constantes que van a ser utilizadas en varios lugares del código.

# Nombre del modelo
modelo_nombre = "CIFAR10_modelo_1"

# Configs para el entrenamiento
epochs = 30
batch_size = 32
learning_rate = 0.001

# Nombres de las clases
class_names = ['avión', 'auto', 'pájaro', 'gato', 'ciervo', 'perro', 'rana', 'caballo', 'bote', 'camión']

In [ ]:
# Cargar y preparar los datos CIFAR-10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [ ]:
# Veamos algunos ejemplos

indices_al_azar = np.random.choice(X_train.shape[0], size=20, replace=False)

imagenes_imagen = X_train[indices_al_azar]
imagenes_etiquetas = y_train[indices_al_azar]

fig, axs = plt.subplots(2, 10, figsize=(10, 3))

for i in range(20):
    row, col = i // 10, i % 10
    axs[row, col].imshow(imagenes_imagen[i])
    axs[row, col].set_title(class_names[imagenes_etiquetas[i][0]])
    axs[row, col].axis("off")

plt.tight_layout()
plt.show()

# Preparar, entrenar y evaluar el modelo

In [ ]:
# Aislar la primera imagen del lote
primera_imagen = X_train[0]
print("\nForma de una sola imagen aislada:", primera_imagen.shape)

In [ ]:
# Inspeccionar los valores de los píxeles
print("\nMatriz de píxeles (RGB) de la primera imagen:")
print(primera_imagen)

In [ ]:
# Ver solo el canal Rojo (R) de esa primera imagen
print("\nSolo el canal ROJO de la primera imagen:")
print(primera_imagen[:, :, 0])

In [ ]:
# Normalizar los píxeles a valores en el rango [0, 1]
X_train, X_test = X_train / 255.0, X_test / 255.0

In [ ]:
# Aislar la primera imagen del lote
primera_imagen = X_train[0]
print("\nForma de una sola imagen aislada:", primera_imagen.shape)

In [ ]:
# Inspeccionar los valores de los píxeles
print("\nMatriz de píxeles (RGB) de la primera imagen:")
print(primera_imagen)

In [ ]:
# Convertir etiquetas a formato one-hot
y_train = tf.keras.utils.to_categorical(y_train, num_classes=10)
y_test = tf.keras.utils.to_categorical(y_test, num_classes=10)

In [ ]:
type(X_train.shape)
X_train.shape[1:]

In [ ]:
# Modelo
modelo = Sequential([
    Input(shape=(32, 32, 3)),
    Conv2D(16, (3, 3), activation='relu', padding='same'),
    Conv2D(16, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    #Dropout(0.2),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    #Dropout(0.2),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    #Dropout(0.2),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    #Dropout(0.2),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

In [ ]:
# Compilar el modelo
modelo.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
              loss=tf.keras.losses.CategoricalCrossentropy(),
              metrics=['accuracy'])

In [ ]:
modelo.summary(line_length=80)

In [ ]:
# Entrenar el modelo
history = modelo.fit(X_train,
                     y_train,
                     epochs=epochs,
                     batch_size=batch_size,
                     # validation_data=(X_train, y_train)
                     )

In [ ]:
# Evaluar el modelo con datos de entrenamiento y de prueba

train_loss, train_acc = modelo.evaluate(X_train, y_train)

test_loss, test_acc = modelo.evaluate(X_test, y_test)

train_loss = round(train_loss, 3)
train_acc = round(train_acc, 3)
test_loss = round(test_loss, 3)
test_acc = round(test_acc, 3)

print(f'Train accuracy: {train_acc}')
print(f'Test accuracy: {test_acc}')

In [ ]:
# Predicciones con el conjunto de prueba.
y_pred_test = modelo.predict(X_test)

In [ ]:
# Clases de las predicciones.
y_pred_test_clase = np.argmax(y_pred_test, axis=1)

# Clases del conjunto de prueba.
y_test_clase = np.argmax(y_test, axis=1)

# Exactitud, Matriz de confusión, Predicciones

In [ ]:
# Curva de entrenamiento

fig, ax = plt.subplots()

fig.suptitle(f"Curva de entrenamiento - Modelo {modelo_nombre}")

ax.plot(history.history['accuracy'], label='Train')

ax.set_title(f"Train ACC: {train_acc}")
ax.set_xticks(np.arange(0, epochs + 1, 2))
ax.set_yticks(np.arange(0, 1.1, 0.1))
ax.set_xlabel('Epoch')
ax.set_ylabel('Exactitud')

ax.legend(loc='lower right')
ax.grid(True)

plt.show()

In [ ]:
def plot_matriz_confusion(matriz_de_confusion,
                          labels = False,
                          titulo = "Matriz de confusión",
                          subtitulo = ""):
    fig, ax = plt.subplots()

    ax = sns.heatmap(matriz_de_confusion,
                     annot=True,
                     cbar=False,
                     fmt ='g')

    if(labels != False):
        ax.set_xticklabels(labels, rotation=45)
        ax.set_yticklabels(labels, rotation=0)

    fig.suptitle(titulo)
    ax.set_title(subtitulo)

    plt.xlabel("Predicción")
    plt.ylabel("Etiqueta real")

    plt.tight_layout()

    plt.show()

In [ ]:
plot_matriz_confusion(confusion_matrix(y_test_clase, y_pred_test_clase),
                      labels=class_names,
                      titulo=f"Matriz de confusión - Modelo {modelo_nombre} - Conjunto Test",
                      subtitulo=f"Accuracy: {test_acc}")

In [ ]:
def preprocess_image(image):
    image = image.resize((32, 32))  # Redimensionar la imagen para ajustarla al tamaño de entrada CIFAR-10
    image = np.array(image)  # Convertir imagen PIL en matriz numpy
    image = image / 255.0  # Normalizar los valores de los píxeles
    return image

In [ ]:
image_urls = [
    'https://img.freepik.com/foto-gratis/tiro-vertical-enfoque-superficial-caballo-marron-que-lleva-arnes-caminando-sobre-suelo-arenoso_181624-22278.jpg',
    'https://img.freepik.com/foto-gratis/disparo-vertical-enfoque-superficial-lindo-cachorro-golden-retriever-sentado-suelo-hierba_181624-27259.jpg',
    'https://img.freepik.com/foto-gratis/gato-rojo-o-blanco-i-estudio-blanco_155003-13189.jpg',
    'https://img.freepik.com/foto-gratis/vista-barco-flotando-agua-paisajes-naturales_23-2150693362.jpg'
]

In [ ]:
for url in image_urls:
    try:
        # Descargar la imagen desde la URL
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Esto levantará una excepción si hay un error en la descarga

        # Cargar la imagen con PIL
        original_image = Image.open(BytesIO(response.content))

        # Guardar una copia de la imagen original para mostrarla
        display_image = original_image.copy()

        # Preprocesar la imagen para el modelo
        processed_image = preprocess_image(original_image)

        # Hacer la predicción
        # Asegurate de que la imagen tenga la forma correcta para tu modelo
        input_image = np.expand_dims(processed_image, axis=0)
        predictions = modelo.predict(input_image)
        predicted_class = np.argmax(predictions)

        # Mostrar la imagen original
        plt.figure(figsize=(6, 6))
        plt.imshow(display_image)
        plt.title(f'Predicción: {class_names[predicted_class]}')
        plt.axis('off')
        plt.show()

    except Exception as e:
        print(f"Error procesando la imagen {url}: {e}")